# D04这份数据是谁

In [ ]:
%reload_ext autoreload
%autoreload 2 
# 上面这俩用于自动重载

import sqlite3 # 先导入内置的模块

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from signal_insight import config, quality, viz, geo
from signal_insight.viz import (MAIN,MAIN_LIGHT,MAIN_DARK,MAIN_MID,GRAY,
                    GRAY_DARK,RED,AMBER,INK,MUTED,FAINT) # 配色常量

con = sqlite3.connect(config.DB_PATH) # 连接数据库
DAY = config.ANALYSIS_DAY
NB = "D03_信令数据洞察"
SOURCE = f"{config.SOURCE_NOTE}:脚本:{NB} 制图:{config.AUTHOR}"

## 封装测试

### 点落面封装测试

In [ ]:
cell = pd.read_sql("SELECT * FROM ods_cell", con)
sites = cell[['lng', 'lat']].drop_duplicates().reset_index(drop=True)
# 给站点做编号：编号补0到S00005
sites['site_id'] = "S" + (sites.index + 1).astype(str).str.zfill(5)

districts = geo.rings_from_points(
    pd.read_sql("SELECT * FROM ods_district_boundary", con), "district")
sites['districts'] = geo.point_in_polygons(sites['lng'], sites['lat'], districts)
sites['districts'] = sites['districts'].fillna("市外")


# 没坐标点扇区定位不了
cell = cell.dropna(subset=['lng','lat']).copy()
# 不重复的坐标=一个站点
# 把站点号贴回每个扇区
cell = cell.merge(sites, on=['lng', 'lat'],how='left')

### 区域地图封装测试

In [ ]:
# 读取dwd表
dwd = pd.read_sql("SELECT * FROM dwd_signal", con, parse_dates=['ts'])
# 两个表格合并
joined = dwd.merge(cell[['lac_cell', "site_id", 'lng', 'lat']],
                on='lac_cell',
                how='left')
out_share = (sites['districts'] == '市外').mean()
rec_district = joined.merge(sites[['site_id', 'districts']],
            on="site_id",how="left")
rec_out = (rec_district['districts'] =='市外').mean()

v2_pts = pd.read_sql("SELECT * FROM ods_aoi_v2", con)
aoi_v2 = geo.rings_from_points(v2_pts,"name")

sites['aoi'] = geo.point_in_polygons(sites['lng'],sites['lat'], aoi_v2)
dim_cell = cell.merge(sites[['site_id', 'aoi']],
                      on='site_id',
                      how='left')
def split_at_jump(values: pd.Series) -> tuple[list[str], list[str], float]:
    """按数值从小到大排，找相邻两项比值最大的地方切开。
    返回 (断崖之前的名字, 断崖之后的名字, 断崖处的倍数)。D3_02、D3_03 用它决定强调哪一组。"""
    v = values.sort_values()
    r = v.values[1:] / v.values[:-1]
    k = int(r.argmax())
    return list(v.index[: k + 1]), list(v.index[k + 1:]), float(r[k])

site = dim_cell.groupby('site_id').agg(
    lng=("lng", 'first'),
    lat=('lat', 'first'),
    district=('districts', 'first')
)

In [ ]:
from scipy.spatial import cKDTree # pip install scipy
# ============================================================
# 2. 经纬度 → 平面公里（沈阳局部平面近似）
# ============================================================
lng0, lat0 = 123.4, 41.8
kx = 111.320 * np.cos(np.radians(lat0))   # 每度经度 ≈ 多少公里
ky = 110.574                               # 每度纬度 ≈ 多少公里
xy = np.column_stack([
    (site["lng"] - lng0) * kx,
    (site["lat"] - lat0) * ky,
]).astype(np.float64)   # cKDTree 用 float64 更稳
# ============================================================
# 3. KD 树求最近邻（等价于原来的分块暴力 + 排除自己）
# ============================================================
tree = cKDTree(xy)
# k=2：第 0 列是自己（距离 0），第 1 列是真正的最近邻
dist_km, idx = tree.query(xy, k=2)
# 取第 1 列作为最近邻距离（公里），转成米
nn = dist_km[:, 1].astype(np.float32)
site["nn_m"] = nn * 1000
# ============================================================
# 4. 各区县面积（鞋带公式，与原代码一致）
# ============================================================
area = {}
for name, rings in districts.items():
    area[name] = sum(
        0.5 * abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))
        for x, y in (((r[:, 0] - lng0) * kx, (r[:, 1] - lat0) * ky) for r in rings)
    )
# ============================================================
# 5. 分组统计
# ============================================================
by = (site[site["district"] != "市外"]
        .groupby("district")
        .agg(站点数=("nn_m", "size"), 间距中位_米=("nn_m", "median"))
        .sort_values("间距中位_米"))
by["每平方公里站点数"] = by["站点数"] / pd.Series(area)
by = by.round(1).reset_index()